# Algorithm Comparison and TSI Closure

Notebook de cierre para comparar Isolation Forest, Local Outlier Factor y DBSCAN con métricas homogéneas de retención, ruido/outliers y estabilidad estructural, y para dejar cerrada la formulación final del Traffic Stability Index (TSI).

In [1]:
from pathlib import Path
import pandas as pd

project_root = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / 'data' / '02_clean').exists()), Path.cwd())
source_path = project_root / 'data' / '02_clean' / 'traffic_enriched.csv'
source_df = pd.read_csv(source_path)
source_count = len(source_df)

results = []

if_path = project_root / 'data' / '02_clean' / 'filtered_isolation_forest.csv'
lof_summary_path = project_root / 'data' / '03_algorithm_output' / 'local_outlier_factor_summary.csv'
dbscan_summary_path = project_root / 'data' / '03_algorithm_output' / 'dbscan_summary.csv'
lof_path = project_root / 'data' / '02_clean' / 'filtered_local_outlier_factor.csv'
dbscan_path = project_root / 'data' / '02_clean' / 'filtered_dbscan.csv'

if_df = pd.read_csv(if_path)
lof_df = pd.read_csv(lof_path)
dbscan_df = pd.read_csv(dbscan_path)
lof_summary = pd.read_csv(lof_summary_path)
dbscan_summary = pd.read_csv(dbscan_summary_path)

results.append({
    'algoritmo': 'Isolation Forest',
    'registros_ret': len(if_df),
    'retencion_%': round(len(if_df) / source_count * 100, 2),
    'ruido_outliers_%': round((source_count - len(if_df)) / source_count * 100, 2),
    'estabilidad_estructural': 'Baja',
    'regimenes_detectados': 1,
    'lectura': 'Excelente depuracion, pero señal estructural limitada.'
})

results.append({
    'algoritmo': 'Local Outlier Factor',
    'registros_ret': len(lof_df),
    'retencion_%': round(float(lof_summary.loc[lof_summary['metric'] == 'porcentaje_retenido', 'value'].iloc[0]), 2),
    'ruido_outliers_%': round(float(lof_summary.loc[lof_summary['metric'] == 'porcentaje_anomalia', 'value'].iloc[0]), 2),
    'estabilidad_estructural': 'Media',
    'regimenes_detectados': 1,
    'lectura': 'Mejor captura vecindad local, pero sigue siendo binario.'
})

results.append({
    'algoritmo': 'DBSCAN',
    'registros_ret': len(dbscan_df),
    'retencion_%': round(float(dbscan_summary.loc[dbscan_summary['metric'] == 'porcentaje_retenido', 'value'].iloc[0]), 2),
    'ruido_outliers_%': round(float(dbscan_summary.loc[dbscan_summary['metric'] == 'porcentaje_ruido', 'value'].iloc[0]), 2),
    'estabilidad_estructural': 'Alta',
    'regimenes_detectados': int(float(dbscan_summary.loc[dbscan_summary['metric'] == 'clusters_detectados', 'value'].iloc[0])),
    'lectura': 'Mejor señal para TSI: separa 6 regímenes densos con ruido moderado.'
})

comparison_df = pd.DataFrame(results).sort_values(
    by=['regimenes_detectados', 'retencion_%'],
    ascending=[False, False],
).reset_index(drop=True)

print(f'Dataset base: {source_path.name} ({source_count} registros)')
print()
print(comparison_df.to_string(index=False))

Dataset base: traffic_enriched.csv (288 registros)

           algoritmo  registros_ret  retencion_%  ruido_outliers_% estabilidad_estructural  regimenes_detectados                                                             lectura
              DBSCAN            262        90.97              9.03                    Alta                     6 Mejor señal para TSI: separa 6 regímenes densos con ruido moderado.
    Isolation Forest            285        98.96              1.04                    Baja                     1              Excelente depuracion, pero señal estructural limitada.
Local Outlier Factor            265        92.01              7.99                   Media                     1            Mejor captura vecindad local, pero sigue siendo binario.


In [2]:
source_df['TSI_propuesto'] = (
    0.45 * source_df['speed_congestion'] +
    0.35 * source_df['densidad_norm'] +
    0.20 * source_df['detenciones_norm']
).clip(0, 1)

if 'TSI' in source_df.columns:
    source_df['TSI_actual'] = source_df['TSI']
    correlation = source_df['TSI_actual'].corr(source_df['TSI_propuesto'])
else:
    correlation = None

comparison_stats = source_df[['TSI_propuesto']].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
print('Formula final del TSI:')
print('TSI = 0.45 * speed_congestion + 0.35 * densidad_norm + 0.20 * detenciones_norm')
print('Rango objetivo: [0, 1] mediante clip')
print()
if correlation is not None:
    print(f'Correlacion con el TSI actual: {correlation:.4f}')
    print()
print('Estadisticas del TSI propuesto:')
print(comparison_stats.to_string())
print()
print('Interpretacion: DBSCAN se usa como validacion estructural; Isolation Forest y LOF ayudan a depurar, pero DBSCAN aporta la senal mas util para definir los regímenes de riesgo.')

Formula final del TSI:
TSI = 0.45 * speed_congestion + 0.35 * densidad_norm + 0.20 * detenciones_norm
Rango objetivo: [0, 1] mediante clip

Correlacion con el TSI actual: 0.9895

Estadisticas del TSI propuesto:
       TSI_propuesto
count     288.000000
mean        0.420025
std         0.227243
min         0.064750
10%         0.160593
25%         0.210904
50%         0.401221
75%         0.607914
90%         0.758021
95%         0.793368
99%         0.851030
max         0.898643

Interpretacion: DBSCAN se usa como validacion estructural; Isolation Forest y LOF ayudan a depurar, pero DBSCAN aporta la senal mas util para definir los regímenes de riesgo.


## Cierre

La comparación homogénea favorece a DBSCAN como señal principal para TSI porque conserva una retención razonable y, sobre todo, revela regímenes densos con ruido moderado. Isolation Forest queda como filtro de depuración y LOF como validación local, pero la formulación final del índice se apoya en una mezcla acotada de congestionamiento, densidad y detenciones.